# OntologySim - Bronze Control

Validation and control checks for the current Bronze ingestion generation.

In [0]:
import json
import uuid

from pyspark.sql import functions as F

generation_id = "8573dcf32f2ed486"
profile = "medium"

landing_base = (
    f"/Volumes/ontologysim/landing/source_files/"
    f"{profile}/{generation_id}"
)

manifest_path = f"{landing_base}/manifest.json"

manifest = json.loads(
    dbutils.fs.head(manifest_path, 100_000)
)

validation_run_id = str(uuid.uuid4())

if manifest["generation_id"] != generation_id:
    raise ValueError("Generation ID mismatch")

if manifest["profile"] != profile:
    raise ValueError("Profile mismatch")

print(f"Generation:     {generation_id}")
print(f"Profile:        {profile}")
print(f"Validation run: {validation_run_id}")

Generation:     8573dcf32f2ed486
Profile:        medium
Validation run: 016c5b65-e978-4aa7-adbe-921d41ad1069


In [0]:
landing_errors = []

for expected_file in manifest["files"]:
    relative_path = expected_file["path"]
    expected_size = expected_file["size_bytes"]

    full_path = f"{landing_base}/{relative_path}"
    parent_path, file_name = full_path.rsplit("/", 1)

    landed_files = {
        item.name.rstrip("/"): item.size
        for item in dbutils.fs.ls(parent_path)
    }

    if file_name not in landed_files:
        landing_errors.append(
            f"Missing file: {relative_path}"
        )
        continue

    actual_size = landed_files[file_name]

    if actual_size != expected_size:
        landing_errors.append(
            f"Size mismatch for {relative_path}: "
            f"expected {expected_size}, got {actual_size}"
        )

if landing_errors:
    raise ValueError("\n".join(landing_errors))

print(
    f"Landing validation passed: "
    f"{len(manifest['files'])} source files verified"
)

Landing validation passed: 10 source files verified


In [0]:
sources = [
    {
        "source": "requirements",
        "manifest_key": "requirements",
        "bronze_table": "ontologysim.bronze.requirements",
        "quarantine_table": None,
    },
    {
        "source": "architecture_components",
        "manifest_key": "components",
        "bronze_table": "ontologysim.bronze.architecture_components",
        "quarantine_table": None,
    },
    {
        "source": "test_management",
        "manifest_key": "test_management",
        "bronze_table": "ontologysim.bronze.test_cases",
        "quarantine_table": None,
    },
    {
        "source": "test_execution",
        "manifest_key": "test_execution",
        "bronze_table": "ontologysim.bronze.test_executions",
        "quarantine_table": None,
    },
    {
        "source": "hcp3_runtime",
        "manifest_key": "hcp3_logs",
        "bronze_table": "ontologysim.bronze.hcp3_runtime",
        "quarantine_table": "ontologysim.quarantine.hcp3_runtime",
    },
    {
        "source": "hcp5_runtime",
        "manifest_key": "hcp5_logs",
        "bronze_table": "ontologysim.bronze.hcp5_runtime",
        "quarantine_table": "ontologysim.quarantine.hcp5_runtime",
    },
    {
        "source": "conmod_runtime",
        "manifest_key": "conmod_logs",
        "bronze_table": "ontologysim.bronze.conmod_runtime",
        "quarantine_table": None,
    },
    {
        "source": "defects",
        "manifest_key": "defects",
        "bronze_table": "ontologysim.bronze.defects",
        "quarantine_table": None,
    },
    {
        "source": "traces",
        "manifest_key": "traces",
        "bronze_table": "ontologysim.bronze.traces",
        "quarantine_table": None,
    },
    {
        "source": "measurements",
        "manifest_key": "measurements",
        "bronze_table": "ontologysim.bronze.measurements",
        "quarantine_table": None,
    },
]

In [0]:
results = []

for source in sources:
    expected_rows = manifest["source_counts"][
        source["manifest_key"]
    ]

    bronze_rows = (
        spark.table(source["bronze_table"])
        .filter(F.col("_generation_id") == generation_id)
        .count()
    )

    quarantine_rows = 0

    if source["quarantine_table"]:
        quarantine_rows = (
            spark.table(source["quarantine_table"])
            .filter(F.col("_generation_id") == generation_id)
            .count()
        )

    accounted_rows = bronze_rows + quarantine_rows

    status = (
        "PASS"
        if accounted_rows == expected_rows
        else "FAIL"
    )

    results.append(
        (
            source["source"],
            expected_rows,
            bronze_rows,
            quarantine_rows,
            accounted_rows,
            status,
        )
    )

validation_df = spark.createDataFrame(
    results,
    [
        "source_family",
        "expected_rows",
        "bronze_rows",
        "quarantine_rows",
        "accounted_rows",
        "status",
    ],
)

display(validation_df)

source_family,expected_rows,bronze_rows,quarantine_rows,accounted_rows,status
requirements,8040,8040,0,8040,PASS
architecture_components,1600,1600,0,1600,PASS
test_management,16080,16080,0,16080,PASS
test_execution,160800,160800,0,160800,PASS
hcp3_runtime,10665216,10643872,21344,10665216,PASS
hcp5_runtime,11429471,11406745,22726,11429471,PASS
conmod_runtime,16497867,16497867,0,16497867,PASS
defects,3216,3216,0,3216,PASS
traces,155976,155976,0,155976,PASS
measurements,7603360,7603360,0,7603360,PASS


In [0]:
failed_sources = (
    validation_df
    .filter(F.col("status") == "FAIL")
    .count()
)

if failed_sources:
    raise ValueError(
        f"{failed_sources} Bronze source validations failed"
    )

print("All Bronze source validations passed")

All Bronze source validations passed


In [0]:
control_df = (
    validation_df
    .withColumn(
        "generation_id",
        F.lit(generation_id),
    )
    .withColumn(
        "profile",
        F.lit(profile),
    )
    .withColumn(
        "validation_run_id",
        F.lit(validation_run_id),
    )
    .withColumn(
        "validated_at",
        F.current_timestamp(),
    )
)

(
    control_df.write
    .format("delta")
    .mode("append")
    .saveAsTable(
        "ontologysim.control.bronze_validation_results"
    )
)

print("Validation results persisted")

Validation results persisted


In [0]:
display(
    spark.table(
        "ontologysim.control.bronze_validation_results"
    )
    .filter(F.col("generation_id") == generation_id)
    .orderBy(
        F.col("validated_at").desc(),
        F.col("source_family"),
    )
)

source_family,expected_rows,bronze_rows,quarantine_rows,accounted_rows,status,generation_id,profile,validation_run_id,validated_at
architecture_components,1600,1600,0,1600,PASS,8573dcf32f2ed486,medium,016c5b65-e978-4aa7-adbe-921d41ad1069,2026-10-14T05:43:50.150Z
conmod_runtime,16497867,16497867,0,16497867,PASS,8573dcf32f2ed486,medium,016c5b65-e978-4aa7-adbe-921d41ad1069,2026-10-14T05:43:50.150Z
defects,3216,3216,0,3216,PASS,8573dcf32f2ed486,medium,016c5b65-e978-4aa7-adbe-921d41ad1069,2026-10-14T05:43:50.150Z
hcp3_runtime,10665216,10643872,21344,10665216,PASS,8573dcf32f2ed486,medium,016c5b65-e978-4aa7-adbe-921d41ad1069,2026-10-14T05:43:50.150Z
hcp5_runtime,11429471,11406745,22726,11429471,PASS,8573dcf32f2ed486,medium,016c5b65-e978-4aa7-adbe-921d41ad1069,2026-10-14T05:43:50.150Z
measurements,7603360,7603360,0,7603360,PASS,8573dcf32f2ed486,medium,016c5b65-e978-4aa7-adbe-921d41ad1069,2026-10-14T05:43:50.150Z
requirements,8040,8040,0,8040,PASS,8573dcf32f2ed486,medium,016c5b65-e978-4aa7-adbe-921d41ad1069,2026-10-14T05:43:50.150Z
test_execution,160800,160800,0,160800,PASS,8573dcf32f2ed486,medium,016c5b65-e978-4aa7-adbe-921d41ad1069,2026-10-14T05:43:50.150Z
test_management,16080,16080,0,16080,PASS,8573dcf32f2ed486,medium,016c5b65-e978-4aa7-adbe-921d41ad1069,2026-10-14T05:43:50.150Z
traces,155976,155976,0,155976,PASS,8573dcf32f2ed486,medium,016c5b65-e978-4aa7-adbe-921d41ad1069,2026-10-14T05:43:50.150Z
